# Analyzing White House Visitor's Logs using Python

First things first, i'll need to import a couple of packages:

In [10]:
from random import randrange
import datetime as dt
from os import listdir
from os.path import isfile, join
from csv import reader

Let's start off by writing a small function to do the reading from the CSV files:

In [11]:
def open_and_read_file(file):
    #Reads the specified csv file and encodes it as a python list of lists
    
	opened_file = open(file, encoding='utf-8', errors='ignore')
	from csv import reader
	read_file = reader(opened_file)
	data = list(read_file)
	return data

## Exploratory Analysis ## 

Our dataset will consist of an aggregation of different CSV files. Before aggregating all the files into a big CSV file and basing my analysis off of that file, i want to do exploratory analysis on only one of the files so that it's easier to handle.

In [12]:
logs_1 = open_and_read_file("../../Datasets/White House Visitor Logs - Obama Administration/VisitorLogs-2012.csv")

The first row of the list of lists logs_1 will be the header of our row. Let's take a look at it:

In [13]:
logs_1_head = logs_1[0]
print(logs_1_head)

['NAMELAST', 'NAMEFIRST', 'NAMEMID', 'UIN', 'BDGNBR', 'ACCESS_TYPE', 'TOA', 'POA', 'TOD', 'POD', 'APPT_MADE_DATE', 'APPT_START_DATE', 'APPT_END_DATE', 'APPT_CANCEL_DATE', 'Total_People', 'LAST_UPDATEDBY', 'POST', 'LastEntryDate', 'TERMINAL_SUFFIX', 'visitee_namelast', 'visitee_namefirst', 'MEETING_LOC', 'MEETING_ROOM', 'CALLER_NAME_LAST', 'CALLER_NAME_FIRST', 'CALLER_ROOM', 'Description', 'Release Date']


This gives me a general idea. Let's print a random row to see it in one piece:

In [14]:
i = 0
for entry in logs_1[randrange(1, len(logs_1))]:
    print(logs_1_head[i] + " is " + entry)
    i += 1

NAMELAST is Winnefeld
NAMEFIRST is James
NAMEMID is A
UIN is U34564
BDGNBR is 0
ACCESS_TYPE is VA
TOA is 8/26/12 16:58
POA is B0402
TOD is 
POD is 
APPT_MADE_DATE is 8/24/12 0:00
APPT_START_DATE is 8/26/12 17:00
APPT_END_DATE is 8/26/12 23:59
APPT_CANCEL_DATE is 
Total_People is 13
LAST_UPDATEDBY is MA
POST is WIN
LastEntryDate is 8/24/12 14:59
TERMINAL_SUFFIX is MA
visitee_namelast is McDonough
visitee_namefirst is Denis
MEETING_LOC is OEOB
MEETING_ROOM is 037 Bowlin
CALLER_NAME_LAST is ALHASSANI
CALLER_NAME_FIRST is MEHDI
CALLER_ROOM is 
Description is location change per alhassani
Release Date is 11/30/12


We can get a better sense of what each column in our dataset stands for now that we have printed out a sample row. It appears to me that some of the columns are there to allow system administrators or other responsibles to debug the faulty consoles when a bug occurs. In light of this, i've determined the columns that are of interest to me as follows:

- NAMELAST --- The last name of the visitor
- NAMEFIRST --- The first name of the visitor
- NAMEMID ---  The middle name of the visitor
- APPT_MADE_DATE ---  Date the appointment was made.
- APPT_START_DATE --- Date the appointment started.
- APPT_END_DATE --- Date the appointment ended
- ACCESS_TYPES --- Type of access to the complex (VA = Visitor Access)
- MEETING_LOC --- Building in which meeting was scheduled
- MEETING_ROOM --- Room in which meeting was scheduled
- VISITEE_NAMELAST --- Last name of the visitee
- VISITEE_NAMEFIRST --- First name of the visitee

The column descriptions are taken from the Obama White House Archives website where the .csv files were downloaded from.
(https://obamawhitehouse.archives.gov/files/disclosures/visitors/WhiteHouse-WAVES-Key-1209.txt)


## Joining the data together

I'll iterate the list of lists so that i could write the columns i want into a list of dictionaries:

In [16]:
columns_of_interest_numbers = [0,1,2,5,10,11,12,14,19,20,21,22]
columns_of_interest_column_names = ["Visitor Last Name",
                                    "Visitor First Name",
                                    "Visitor Middle Name",
                                    "Access Type",
                                    "Appointment Creation Date",
                                    "Appointment Start Date",
                                    "Appointment End Date",
                                    "Total People in Appointment",
                                    "Visitee Last Name",
                                    "Visitee First Name",
                                    "Meeting Location(Building)",
                                    "Meeting Room",
                                    ]

In [21]:
logs_1_list_of_dicts = []

for row in logs_1:
    dictionary = {}
    for num, name in zip(columns_of_interest_numbers, columns_of_interest_column_names):
        dictionary[name] = row[num]
    logs_1_list_of_dicts.append(dictionary)
    
print(logs_1_list_of_dicts[randrange(1,len(logs_1_list_of_dicts))])

{'Visitor Last Name': 'Kretkowski', 'Visitor First Name': 'Ronald', 'Visitor Middle Name': 'C', 'Access Type': 'VA', 'Appointment Creation Date': '2/17/12 0:00', 'Appointment Start Date': '2/25/12 9:30', 'Appointment End Date': '2/25/12 23:59', 'Total People in Appointment': '247', 'Visitee Last Name': 'OFFICE', 'Visitee First Name': 'VISITORS', 'Meeting Location(Building)': 'WH', 'Meeting Room': 'RESIDENCE'}


That worked. Now, i need to do this for all the files in the directory where the .CSV files are located. This function would work if i were to pass all rows into it:

In [22]:
logs_list_of_dict = []

def lst_entry_into_dict(lst):
    for row in lst:
        dictionary = {}
        for num, name in zip(columns_of_interest_numbers, columns_of_interest_column_names):
            dictionary[name] = row[num]
        logs_list_of_dict.append(dictionary)

How can i read all the files in a directory? The code block below just might do it:

In [23]:
mypath = "../../Datasets/White House Visitor Logs - Obama Administration"
onlyfiles = [f for f in listdir(mypath) if isfile(join(mypath, f))]

for file in onlyfiles:
    with open(mypath + "/" + file, encoding='utf-8', errors='ignore' ) as csv:
        read_file = reader(csv)
        lst_entry_into_dict(list(read_file))
        csv.close()

In [24]:
len(logs_list_of_dict)

3341154

Okay, now we have one big list of dictionaries we can base our analysis on.

## Data Cleaning ##

### Initial Cleaning ###

Now, what i want to do is to take a random sample on N size from my dataset so that i can see the overall features of the entries. The below code would do it for random sampling:

In [26]:
def random_sampling(population,sample_size):
    # Takes n(sample_size) samples from the population.
    #returns a list containing all the samples
    samples = []
    for i in range (0,sample_size):
        random = randrange(1, len(population))
        samples.append(population[random])
    return samples
        
samples = random_sampling(logs_list_of_dict,5)


We also need a function to display the entries along with the column name to be able to make a sense of it:

In [27]:
def explore_columns(dataset):
    header_example = dataset[0]
    headers = [header for header in header_example.keys()]
    for header in headers:
        print(header)
        for item in dataset:
            print(item[header])
        print("")

In [31]:
samples = random_sampling(logs_list_of_dict,5)
explore_columns(samples)

Visitor Last Name
Reed
ANDERSON
Feinstein
Tate
Dooley

Visitor First Name
Deborah
LOGAN
Kim
Katharina
Brendon

Visitor Middle Name
S
D
H
M
H

Access Type
VA
VA
VA
VA
VA

Appointment Creation Date
3/10/2016 0:00
6/30/2016 0:00
4/29/2013 0:00
11/18/2014 0:00
4/18/12 0:00

Appointment Start Date
3/19/2016 8:30
7/7/2016 8:30
5/1/2013 13:00
11/30/2014 17:30
4/18/12 15:00

Appointment End Date
3/19/2016 23:59
7/7/2016 23:59
5/1/2013 23:59
11/30/2014 23:59
4/18/12 23:59

Total People in Appointment
274
16
56
5
13

Visitee Last Name
OFFICE
Office
Thompson
Addarich
Cho

Visitee First Name
VISITORS
Visitors
Jim
Raymond
Ronnie

Meeting Location(Building)
WH
WH
OEOB
WH
OEOB

Meeting Room
EW TOUR
EW TOUR
430
WEST WING
430A



Okay, we have discovered some potential problems and irregularities regarding the entries in our dataset. The ones that i will spend time trying to correct are as follows:
 
 - Visitor/Visitee first names, middle names and last names follow a mixed casing format.
 - Appointment creation dates:
     - Are reliable only to the day, as all hours are logged in as 00:00
     - Generally follow a M/D/Y format, however there appears to be some exceptions.
 - Appointment end dates are reliable only to the day, as all hours are logged in as 00:00
 
I will address these issues seperately. 

#### Casing conversion

This is a pretty straightforward function utilizing the out - of - the - box string manipulation methods.

In [34]:
def convert_casing(dataset,relevant_columns,conversion_type):
    #Accepts a list of dicts as dataset.
    #Accepts a list of strings as relevant columns.
    #Converts all selected columns into the specified format determined by conversion_type
    
    for row in dataset:
        for column in relevant_columns:
            entry = row[column]
            if conversion_type == "Uppercase":
                entry = entry.upper()
                row[column] = entry
            elif conversion_type == "Lowercase":
                entry = entry.lower(),
                row[column] = entry
            elif conversion_type == "Capitalize":
                entry = entry.capitalize()
                row[column] = entry

#### Timedate conversion

This function uses a lot of repetitive flow control to account for the existence of different time formats in the dataset. **I acknowledge that this might not be the most elegant or effective way to write this function.** 

In [35]:
def string_to_timedate(dataset, relevant_columns):
    #Accepts a list of dicts as a dataset.
    #Accepts a list of strings as relevent columns
    #Converts all selected columns into a timedate format.
    for row in dataset:
        for column in relevant_columns:
            entry = row[column]
            
            if len(entry) == 0:
                row[column] = "None"
                
            elif "/" not in entry:
                row[column] = "Not a valid date"
                
            else:
                try:
                    date, time = entry.split()
                    month, day, year = date.split("/")
                    if len(year) == 4 and int(month) <= 12:
                        datetime = dt.datetime.strptime(entry, "%m/%d/%Y %H:%M")
                    elif len(year) == 4 and int(month) > 12:
                        datetime_string = dt.datetime.strptime(entry, "%d/%m/%Y %H:%M").strftime("%m/%d/%Y %H:%M")
                        #The above code is hackey way to deal with differing formats.
                        datetime = dt.datetime.strptime(datetime_string, "%m/%d/%Y %H:%M")
                    else:
                        datetime = dt.datetime.strptime(entry, "%m/%d/%y %H:%M")
                    row[column] = datetime
                except:
                    date = entry
                    month, day, year = date.split("/")
                    if len(year) == 4 and int(month) <= 12:
                        datetime = dt.datetime.strptime(entry, "%m/%d/%Y")
                                                        
                    elif len(year) == 4 and int(month) > 12:
                        datetime_string = dt.datetime.strptime(entry, "%d/%m/%Y").strftime("%m/%d/%Y %H:%M")
                        #The above code is hackey way to deal with differing formats.
                        datetime = dt.datetime.strptime(datetime_string, "%m/%d/%Y %H:%M")
                    
                    if len(year) == 2 and int(month) <= 12:
                        datetime_string = dt.datetime.strptime(entry, "%m/%d/%y").strftime("%m/%d/%Y %H:%M")
                        datetime = dt.datetime.strptime(datetime_string, "%m/%d/%Y %H:%M")                                     
                    elif len(year) == 2 and int(month) > 12:
                        datetime_string = dt.datetime.strptime(entry, "%d/%m/%y").strftime("%m/%d/%Y %H:%M")
                        #The above code is hackey way to deal with differing formats.
                        datetime = dt.datetime.strptime(datetime_string, "%m/%d/%Y %H:%M")
                    row[column] = datetime

Now, let's use the functions above to clean our data:

In [36]:
relevant_columns_names = ["Visitor Last Name","Visitor First Name",
                          "Visitor Middle Name","Visitee Last Name","Visitee First Name"]
relevant_columns_dates = ["Appointment Creation Date", "Appointment Start Date", "Appointment End Date"]

string_to_timedate(logs_list_of_dict, relevant_columns_dates)
convert_casing(logs_list_of_dict,relevant_columns_names, "Uppercase")

### Checking for cleaning errors

Since the methods i've used to check for potential problems and clean them was far from perfect, it would be a smart idea to check to see whether the cleaning methods i used have succeeded. For this, let's take two random samples and print them to see if we can catch any problems:

In [37]:
sample_1 = random_sampling(logs_list_of_dict,5)
sample_2 = random_sampling(logs_list_of_dict,5)

explore_columns(sample_1)
explore_columns(sample_2)

Visitor Last Name
HETRICH
BIENENFELD
LEE
MARTIN
RAMDAT

Visitor First Name
MARK
ROBERT
TIYING
ERIK
REGINALD

Visitor Middle Name
H
J

N
N

Access Type
VA
VA

VA
VA

Appointment Creation Date
2016-05-05 00:00:00
2013-03-20 00:00:00
None
2015-01-21 00:00:00
2016-06-15 00:00:00

Appointment Start Date
2016-05-14 12:30:00
2013-03-20 15:00:00
2013-12-14 00:00:00
2015-01-21 12:30:00
2016-06-22 07:30:00

Appointment End Date
2016-05-14 23:59:00
2013-03-20 23:59:00
None
2015-01-21 23:59:00
2016-06-22 23:59:00

Total People in Appointment
263
3

1
3

Visitee Last Name
WAVES
WHITEMAN

JENCKS
OFFICE

Visitee First Name
VISITORSOFFICE
CHAD

FAE
VISITORS

Meeting Location(Building)
WH
NEOB
VPR
OEOB
WH

Meeting Room
EW TOUR
NEOB Eater

SCA
EW TOUR

Visitor Last Name
RICHARD
BERNAL
BENTLEY
SERRANT
SIMONTHOMPSON

Visitor First Name
NATALIE
CARLA
LAUREN
NICOLE
BETTYANN

Visitor Middle Name
M
V
G
D
J

Access Type
VA
VA
VA
VA
VA

Appointment Creation Date
2016-08-04 00:00:00
2015-02-10 00:00:00
2013-12-1

Although it seems to be fine, i think the datetime columns warrant for a more methodical investigation. See below:

In [47]:
def track_faulty_row(dataset,relevant_columns,type_to_check_for):
    #Accepts a list of dicts, a list and a class name as inputs for parameters.
    #Checks if the given columns of a row belong to the specified class
    #If not, updates an total error counter and logs in the index of the row.
    i = 0
    errors_list = []
    error_count = 0
    for row in dataset:
        i += 1
        for column in relevant_columns:
            entry = row[column]
            if isinstance(entry, type_to_check_for) != True:
                dictionary = {}
                dictionary["Row Number"] = i
                dictionary["Column Name"] = str(column)
                errors_list.append(dictionary)
                error_count = len(errors_list)
            else:
                pass
    return error_count, errors_list
                
results = track_faulty_row(logs_list_of_dict, relevant_columns_dates,dt.datetime)

In [51]:
results = track_faulty_row(logs_list_of_dict, relevant_columns_dates,dt.datetime)
results[1][randrange(1,len(results[1]))] # This marks row num. 1442403 as faulty
logs_list_of_dict[1442403]["Appointment End Date"]

'Not a valid date'

Alright, it appears that i accounted for all the possible errors that can occur during datetime conversion! Let's move on.

## Data Analysis ## 

As i stated above, i will create a series of frequency distributions to observe the yearly, monthly, daily and hourly distribution of meeting location and meeting date. The questions that i will be asking are as follows:

 - What is the frequency table for meeting location over all years and for each individual year?
 - What is the frequency table for meeting room over all years and for each individual year?
 - What is the frequency table for meeting start date year, month, days and hours for all years?
 - What is the frequency table for meeting start hour for each individual month?
 
**What excites me most is the last frequency table. By visualizing that frequency table, we might be able to get to see whether there is a difference in between different months and different seasons (summer vs. winter) regarding appointment start hours.**
 

### 1ST

In [54]:
meeting_location_frequency_table_all = {}
meeting_location_frequency_table_2012 = {}
meeting_location_frequency_table_2013 = {}
meeting_location_frequency_table_2014 = {}
meeting_location_frequency_table_2015 = {}
meeting_location_frequency_table_2016 = {}

for row in logs_list_of_dict:
    try:
        year_check = row["Appointment Start Date"].year
        if year_check == 2012:
            if row["Meeting Location(Building)"] not in meeting_location_frequency_table_2012:
                meeting_location_frequency_table_2012[row["Meeting Location(Building)"]] = 1
            else:
                meeting_location_frequency_table_2012[row["Meeting Location(Building)"]] += 1
        if year_check == 2013:
            if row["Meeting Location(Building)"] not in meeting_location_frequency_table_2013:
                meeting_location_frequency_table_2013[row["Meeting Location(Building)"]] = 1
            else:
                meeting_location_frequency_table_2013[row["Meeting Location(Building)"]] += 1
        if year_check == 2014:
             if row["Meeting Location(Building)"] not in meeting_location_frequency_table_2014:
                meeting_location_frequency_table_2014[row["Meeting Location(Building)"]] = 1
             else:
                meeting_location_frequency_table_2014[row["Meeting Location(Building)"]] += 1
        if year_check == 2015:
             if row["Meeting Location(Building)"] not in meeting_location_frequency_table_2015:
                meeting_location_frequency_table_2015[row["Meeting Location(Building)"]] = 1
             else:
                meeting_location_frequency_table_2015[row["Meeting Location(Building)"]] += 1
        if year_check == 2016:
             if row["Meeting Location(Building)"] not in meeting_location_frequency_table_2016:
                meeting_location_frequency_table_2016[row["Meeting Location(Building)"]] = 1
             else:
                meeting_location_frequency_table_2016[row["Meeting Location(Building)"]] += 1
        if row["Meeting Location(Building)"] not in meeting_location_frequency_table_all:
            meeting_location_frequency_table_all[row["Meeting Location(Building)"]] = 1
        else:
            meeting_location_frequency_table_all[row["Meeting Location(Building)"]] += 1
    except:
        continue

### 2nd

In [ ]:
meeting_room_frequency_table_all = {}
meeting_room_frequency_table_2012 = {}
meeting_room_frequency_table_2013 = {}
meeting_room_frequency_table_2014 = {}
meeting_room_frequency_table_2015 = {}
meeting_room_frequency_table_2016 = {}        

for row in logs_list_of_dict:
    try:
        year_check = row["Appointment Start Date"].year
        if year_check == 2012:
            if row["Meeting Location(Building)"] not in meeting_room_frequency_table_2012:
                meeting_room_frequency_table_2012[row["Meeting Room"]] = 1
            else:
                meeting_room_frequency_table_2012[row["Meeting Room"]] += 1
        if year_check == 2013:
            if row["Meeting Room"] not in meeting_room_frequency_table_2013:
                meeting_room_frequency_table_2013[row["Meeting Room"]] = 1
            else:
                meeting_room_frequency_table_2013[row["Meeting Room"]] += 1
        if year_check == 2014:
             if row["Meeting Room"] not in meeting_room_frequency_table_2014:
                meeting_room_frequency_table_2014[row["Meeting Room"]] = 1
             else:
                meeting_room_frequency_table_2014[row["Meeting Room"]] += 1
        if year_check == 2015:
             if row["Meeting Room"] not in meeting_room_frequency_table_2015:
                meeting_room_frequency_table_2015[row["Meeting Room"]] = 1
             else:
                meeting_room_frequency_table_2015[row["Meeting Room"]] += 1
        if year_check == 2016:
             if row["Meeting Room"] not in meeting_room_frequency_table_2016:
                meeting_room_frequency_table_2016[row["Meeting Room"]] = 1
             else:
                meeting_room_frequency_table_2016[row["Meeting Room"]] += 1
        if row["Meeting Room"] not in meeting_room_frequency_table_all:
            meeting_room_frequency_table_all[row["Meeting Room"]] = 1
        else:
            meeting_room_frequency_table_all[row["Meeting Room"]] += 1
    except:
        continue

### 3rd

In [ ]:
#Distribution of meetings by year
meeting_start_years_frequency = {}
#Distribution of meetings by month
meeting_start_months_frequency = {}
#Distribution of meetings by day
meeting_start_days_frequency = {}
#Distribution of meetings by hour
meeting_start_hours_frequency = {}       

for row in logs_list_of_dict:
    try:
        year_check = row["Appointment Start Date"].year
        
        if row["Appointment Start Date"].year not in meeting_start_years_frequency:
            meeting_start_years_frequency[row["Appointment Start Date"].year] = 1
        else:
            meeting_start_years_frequency[row["Appointment Start Date"].year] += 1
        
        if row["Appointment Start Date"].month not in meeting_start_months_frequency:
            meeting_start_months_frequency[row["Appointment Start Date"].month] = 1
        else:
            meeting_start_months_frequency[row["Appointment Start Date"].month] += 1
            
        if row["Appointment Start Date"].day not in meeting_start_days_frequency:
            meeting_start_days_frequency[row["Appointment Start Date"].day] = 1
        else:
            meeting_start_days_frequency[row["Appointment Start Date"].day] += 1
            
        if row["Appointment Start Date"].hour not in meeting_start_hours_frequency:
            meeting_start_hours_frequency[row["Appointment Start Date"].hour] = 1
        else:
            meeting_start_hours_frequency[row["Appointment Start Date"].hour] += 1
            
    except:
        continue

### 4th

In [55]:
meeting_start_hours_frequency_01 = {}
meeting_start_hours_frequency_02 = {}
meeting_start_hours_frequency_03 = {}
meeting_start_hours_frequency_04 = {}
meeting_start_hours_frequency_05 = {}
meeting_start_hours_frequency_06 = {}
meeting_start_hours_frequency_07 = {}
meeting_start_hours_frequency_08 = {}
meeting_start_hours_frequency_09 = {}
meeting_start_hours_frequency_10 = {}
meeting_start_hours_frequency_11 = {}
meeting_start_hours_frequency_12 = {}

for row in logs_list_of_dict:
    
    try:
        month_check = row["Appointment Start Date"].month
        
        if month_check == 1:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_01:
                meeting_start_hours_frequency_01[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_01[row["Appointment Start Date"].hour] += 1 
                
        if month_check == 2:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_02:
                meeting_start_hours_frequency_02[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_02[row["Appointment Start Date"].hour] += 1
                
        if month_check == 3:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_03:
                meeting_start_hours_frequency_03[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_03[row["Appointment Start Date"].hour] += 1
                
        if month_check == 4:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_04:
                meeting_start_hours_frequency_04[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_04[row["Appointment Start Date"].hour] += 1
                
        if month_check == 5:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_05:
                meeting_start_hours_frequency_05[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_05[row["Appointment Start Date"].hour] += 1
                
        if month_check == 6:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_06:
                meeting_start_hours_frequency_06[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_06[row["Appointment Start Date"].hour] += 1
                
        if month_check == 7:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_07:
                meeting_start_hours_frequency_07[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_07[row["Appointment Start Date"].hour] += 1
                
        if month_check == 8:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_08:
                meeting_start_hours_frequency_08[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_08[row["Appointment Start Date"].hour] += 1
                
        if month_check == 9:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_09:
                meeting_start_hours_frequency_09[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_09[row["Appointment Start Date"].hour] += 1
                
        if month_check == 10:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_10:
                meeting_start_hours_frequency_10[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_10[row["Appointment Start Date"].hour] += 1
                
        if month_check == 11:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_11:
                meeting_start_hours_frequency_11[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_11[row["Appointment Start Date"].hour] += 1
                
        if month_check == 12:
            if row["Appointment Start Date"].hour not in meeting_start_hours_frequency_12:
                meeting_start_hours_frequency_12[row["Appointment Start Date"].hour] = 1
            else:
                meeting_start_hours_frequency_12[row["Appointment Start Date"].hour] += 1 
    except:
        continue